<a href="https://colab.research.google.com/github/nguyenhunganh006/RMSD/blob/main/RMSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install CondaColab to manage RDKit, OpenMM, and MDAnalysis
!pip install -q condacolab
import condacolab
condacolab.install()

# Install OpenMM and analysis tools
!mamba install -c conda-forge openmm pdbfixer mdanalysis matplotlib

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:12
🔁 Restarting kernel...

Looking for: ['openmm', 'pdbfixer', 'mdanalysis', 'matplotlib']

[+] 0.0s
conda-forge/linux-64  ⣾  [+] 0.1s
conda-forge/linux-64   1%
conda-forge/noarch     1%[+] 0.2s
conda-forge/linux-64   8%
conda-forge/noarch    17%[+] 0.3s
conda-forge/linux-64  14%
conda-forge/noarch    31%[+] 0.4s
conda-forge/linux-64  19%
conda-forge/noarch    40%[+] 0.5s
conda-forge/linux-64  24%
conda-forge/noarch    49%[+] 0.6s
conda-forge/linux-64  27%
conda-forge/noarch    56%[+] 0.7s
conda-forge/linux-64  29%
conda-forge/noarch    61%[+] 0.8s
conda-forge/linux-64  35%
conda-forge/noarch    72%[+] 0.9s
conda-forge/linux-64  37%
conda-forge/noarch    78%[+] 1.0s
conda-forge/linux-64  39%
conda-forge/noarch    81%[+] 1.1s
conda-forge/linux-64  41%
conda-forge/noa

In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
import os

# --- Configuration ---
pdb_file = 'complex.pdb'
psf_file = 'complex.psf'
output_dcd = 'production_50ns.dcd'
checkpoint_file = 'checkpoint.chk'
total_steps = 25000000 # 50 ns at 2fs timestep
report_freq = 25000    # Save frame every 50 ps (1,000 frames total)
checkpoint_freq = 500000 # Save checkpoint every 1 ns

# --- System Setup (CHARMM36) ---
psf = CharmmPsfFile(psf_file)
pdb = PDBFile(pdb_file)

# Load CHARMM parameters
params = CharmmParameterSet('par_all36_prot.prm', 'par_all36_lipid.prm', 'toppar_water_ions.str', 'chi.prm')

system = psf.createSystem(params, nonbondedMethod=PME,
                          nonbondedCutoff=1.2*nanometers,
                          constraints=HBonds)

# Integrator: Langevin Middle Integrator for better temperature control
integrator = LangevinMiddleIntegrator(310*kelvin, 1/picosecond, 0.002*picoseconds)

# Barostat: Maintain 1 atm pressure (NPT Ensemble)
system.addForce(MonteCarloBarostat(1*bar, 310*kelvin))

# --- Simulation Initialization ---
platform = Platform.getPlatformByName('CUDA') # Use GPU
simulation = Simulation(psf.topology, system, integrator, platform)

# Resume from Checkpoint if it exists
if os.path.exists(checkpoint_file):
    print("Resuming from checkpoint...")
    with open(checkpoint_file, 'rb') as f:
        simulation.context.loadCheckpoint(f.read())
else:
    print("Starting new simulation...")
    simulation.context.setPositions(pdb.positions)
    simulation.minimizeEnergy()
    simulation.context.setVelocitiesToTemperature(310*kelvin)

# --- Reporters (Output) ---
simulation.reporters.append(DCDReporter(output_dcd, report_freq, append=True))
simulation.reporters.append(StateDataReporter(sys.stdout, report_freq, step=True,
                            potentialEnergy=True, temperature=True, speed=True))
simulation.reporters.append(CheckpointReporter(checkpoint_file, checkpoint_freq))

# --- Run Production ---
print(f"Running {total_steps} steps...")
simulation.step(total_steps)
print("Simulation Complete!")
Step 3: Rapid Convergence Analysis (RMSD)
Once the 50 ns run is complete, use this block to generate the RMSD plot for your PNAS manuscript. This proves the stability of the CHI-PHD2 complex.

Python
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import matplotlib.pyplot as plt

# Load the trajectory
u = mda.Universe('complex.psf', 'production_50ns.dcd')
ref = mda.Universe('complex.psf', 'production_50ns.dcd') # Reference frame 0

# Calculate RMSD for Protein Backbone and CHI Ligand
R_prot = rms.RMSD(u, ref, select="backbone").run()
R_chi = rms.RMSD(u, ref, select="resname CHI").run()

# Plotting
time = R_prot.results.rmsd[:, 1] * 0.05 # Convert saved frames to ns (50ps/frame)
rmsd_prot = R_prot.results.rmsd[:, 2]
rmsd_chi = R_chi.results.rmsd[:, 2]

plt.figure(figsize=(8, 5))
plt.plot(time, rmsd_prot, label="Protein Backbone", color='black')
plt.plot(time, rmsd_chi, label="CHI Ligand", color='red', alpha=0.7)
plt.xlabel("Time (ns)", fontsize=12)
plt.ylabel(r"RMSD ($\AA$)", fontsize=12)
plt.title("Molecular Dynamics Stability (50 ns)", fontsize=14)
plt.legend()
plt.grid(True, linestyle='--')
plt.savefig('RMSD_Convergence.png', dpi=300)
plt.show()

In [ ]:
# 1. Install CondaColab for environment management
!pip install -q condacolab
import condacolab
condacolab.install()

# 2. Install OpenMM, OpenFF, PDBFixer, and Analysis tools
!mamba install -c conda-forge openmm openmmforcefields openff-toolkit pdbfixer mdanalysis matplotlib -y

In [ ]:
import os
import sys
from openmm import *
from openmm.app import *
from openmm.unit import *
from openmmforcefields.generators import SystemGenerator
from openff.toolkit.topology import Molecule
from pdbfixer import PDBFixer

def run_phd_simulation(protein_name, pdb_input, ligand_sdf, nanoseconds=50):
    print(f"--- Initializing Simulation for {protein_name} ---")

    # 1. Prepare Ligand using OpenFF
    molecule = Molecule.from_file(ligand_sdf)

    # 2. Fix Protein Structure (Add missing atoms/hydrogens)
    fixer = PDBFixer(filename=pdb_input)
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.4)

    # 3. Setup Forcefield (Amber14SB for protein, OpenFF 2.0 'Sage' for ligand)
    forcefield_kwargs = {'constraints': HBonds, 'rigidWater': True, 'removeCMMotion': False, 'hydrogenMass': 4*amu}
    periodic_kwargs = {'nonbondedMethod': PME, 'nonbondedCutoff': 1.0*nanometer}

    system_generator = SystemGenerator(
        forcefields=['amber14-all.xml', 'amber14/tip3p.xml'],
        small_molecule_forcefield='openff-2.0.0',
        molecules=[molecule],
        cache='db.json',
        forcefield_kwargs=forcefield_kwargs,
        periodic_forcefield_kwargs=periodic_kwargs
    )

    # 4. Combine Protein and Ligand
    # We assume the PDB already contains the docked ligand for simplicity
    modeller = Modeller(fixer.topology, fixer.positions)
    modeller.addSolvent(system_generator.forcefield, padding=1.2*nanometers, ionicStrength=0.15*molar)

    # 5. Create System and Simulation
    system = system_generator.create_system(modeller.topology)
    system.addForce(MonteCarloBarostat(1*bar, 310*kelvin))
    integrator = LangevinMiddleIntegrator(310*kelvin, 1/picosecond, 0.002*picoseconds)

    platform = Platform.getPlatformByName('CUDA')
    sim = Simulation(modeller.topology, system, integrator, platform)
    sim.context.setPositions(modeller.positions)

    # 6. Minimization and Equilibration
    print(f"Minimizing {protein_name}...")
    sim.minimizeEnergy()
    sim.context.setVelocitiesToTemperature(310*kelvin)

    # 7. Production Run (50 ns)
    dcd_file = f"{protein_name}_prod.dcd"
    checkpoint = f"{protein_name}.chk"

    sim.reporters.append(DCDReporter(dcd_file, 50000)) # Save frame every 100ps
    sim.reporters.append(StateDataReporter(sys.stdout, 50000, step=True, potentialEnergy=True, temperature=True, speed=True))
    sim.reporters.append(CheckpointReporter(checkpoint, 500000)) # Checkpoint every 1ns

    steps = int((nanoseconds * nanosecond) / (0.002 * picosecond))
    print(f"Starting {nanoseconds}ns Production for {protein_name}...")
    sim.step(steps)
    print(f"--- Finished {protein_name} ---")

# --- MAIN EXECUTION LOOP ---
complexes = {
    "PHD1": "PHD1_docked.pdb",
    "PHD2": "PHD2_docked.pdb",
    "PHD3": "PHD3_docked.pdb"
}

for name, pdb in complexes.items():
    if os.path.exists(pdb):
        run_phd_simulation(name, pdb, "chi.sdf", nanoseconds=50)
    else:
        print(f"Error: {pdb} not found. Skipping {name}.")

In [1]:
import MDAnalysis as mda
from MDAnalysis.analysis import rms
import matplotlib.pyplot as plt

def analyze_rmsd(name, psf_like_pdb, dcd):
    u = mda.Universe(psf_like_pdb, dcd)
    ref = mda.Universe(psf_like_pdb, dcd) # Use first frame as reference
    R = rms.RMSD(u, ref, select="backbone").run()
    return R.results.rmsd[:, 1] * 0.1, R.results.rmsd[:, 2] # Time in ns, RMSD in A

plt.figure(figsize=(10, 6))
colors = {"PHD1": "blue", "PHD2": "red", "PHD3": "green"}

for name in ["PHD1", "PHD2", "PHD3"]:
    dcd = f"{name}_prod.dcd"
    pdb = f"{name}_docked.pdb"
    if os.path.exists(dcd):
        time, val = analyze_rmsd(name, pdb, dcd)
        plt.plot(time, val, label=f"CHI-{name}", color=colors[name])

plt.xlabel("Time (ns)")
plt.ylabel(r"Backbone RMSD ($\AA$)")
plt.title("Comparative Stability of CHI across PHD Isoforms")
plt.legend()
plt.savefig("Isoform_Selectivity_RMSD.png", dpi=300)
plt.show()

ModuleNotFoundError: No module named 'MDAnalysis'